In [3]:
# Environment + quick dataset listing (Kaggle/Colab/local)

import os
from pathlib import Path

KAGGLE_INPUT = Path('/kaggle/input')
if KAGGLE_INPUT.exists():
    print('Kaggle input detected:', KAGGLE_INPUT)
    for p in KAGGLE_INPUT.rglob('*'):
        if p.is_file():
            print(p)
else:
    print('Not running on Kaggle. Current working directory:')
    print(Path.cwd())

# Tip: you can set dataset paths via env vars used below:
#   SUGARCANE_DATASET_SRC, SUGARCANE_DATASET_DST


In [2]:
import os
import shutil
from pathlib import Path

# Kaggle dataset layout (default). Override via environment variables if needed.
base_path = Path(os.environ.get('SUGARCANE_DATASET_SRC', '/kaggle/input/sugarcane-leaf-disease-dataset'))
target_path = Path(os.environ.get('SUGARCANE_DATASET_DST', '/kaggle/working/sugarcane_dataset'))

if not base_path.exists():
    print(f'[SKIP] Dataset source not found: {base_path}')
    print('Set SUGARCANE_DATASET_SRC to your dataset folder path and re-run this cell.')
else:
    # Delete existing directories before copying
    folders_to_delete = ['Healthy', 'Unhealthy']
    for folder in folders_to_delete:
        folder_path = target_path / folder
        if folder_path.exists():
            shutil.rmtree(folder_path)
            print(f'Deleted existing directory: {folder_path}')

    # Create new directories
    healthy_dst = target_path / 'Healthy'
    unhealthy_dst = target_path / 'Unhealthy'

    healthy_dst.mkdir(parents=True, exist_ok=True)
    unhealthy_dst.mkdir(parents=True, exist_ok=True)

    # Copy Healthy images
    healthy_src = base_path / 'Healthy'
    for file in healthy_src.glob('*'):
        if file.is_file():
            shutil.copy2(file, healthy_dst / file.name)

    # Copy Unhealthy images with folders
    unhealthy_categories = ['Mosaic', 'RedRot', 'Rust', 'Yellow']
    for category in unhealthy_categories:
        cat_src = base_path / category
        cat_dst = unhealthy_dst / category
        cat_dst.mkdir(parents=True, exist_ok=True)
        for file in cat_src.glob('*'):
            if file.is_file():
                shutil.copy2(file, cat_dst / file.name)

    print('Dataset copied with new folder structure successfully.')
    print('Target:', target_path)


Dataset copied with new folder structure successfully.


In [ ]:
# Step 1: Install compatible versions
# Removed tensorflow-model-optimization due to compatibility issues
import sys
import subprocess
import importlib.util

if importlib.util.find_spec('tensorflow') is None:
    # In Kaggle this should be already installed; in Colab/local it may install.
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'tensorflow'])


# # Step 2: Import libraries
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
# Removed tensorflow_model_optimization as tfmot
import numpy as np
import os
import glob
from PIL import Image
import matplotlib.pyplot as plt

print(f"TensorFlow version: {tf.__version__}")
print(f"Keras version: {tf.keras.__version__}")

DATASET_DIR = os.environ.get('SUGARCANE_DATASET_DST', '/kaggle/working/sugarcane_dataset')

# Step 3: Load and preprocess dataset
def load_and_preprocess_dataset():
    path = DATASET_DIR
    print("Path to dataset files:", path)
    print(f"Does path exist: {os.path.exists(path)}")


    classes = ['Healthy', 'Unhealthy']

    def load_dataset(data_dir, classes, target_size=(256, 256)):
        images = []
        labels = []

        for class_idx, class_name in enumerate(classes):
            if class_name == 'Healthy':
                class_path_pattern = os.path.join(data_dir, 'Healthy', '*.jpeg')
            else:
                class_path_pattern = os.path.join(data_dir, 'Unhealthy', '*', '*.jpeg')


            print(f"Looking for files with pattern: {class_path_pattern}")
            image_files = glob.glob(class_path_pattern)
            print(f"Found {len(image_files)} images for class {class_name}")

            for img_path in image_files:
                try:
                    img = Image.open(img_path).convert('RGB')
                    # Resize image to target_size
                    img_resized = img.resize(target_size, Image.Resampling.LANCZOS)
                    img_array = np.array(img_resized)


                    # Skip images with 4 dimensions (e.g., RGBA) and check for 3 dimensions
                    if len(img_array.shape) != 3:
                         print(f"Skipping image {img_path} with unexpected shape after resizing: {img_array.shape}")
                         continue
                    # Optional: Add a check for expected shape, e.g., (height, width, 3)
                    # if img_array.shape[-1] != 3:
                    #     print(f"Skipping image {img_path} with unexpected channel dimension: {img_array.shape}")
                    #     continue


                    images.append(img_array)
                    labels.append(class_idx)
                except Exception as e:
                    print(f"Error loading or processing image {img_path}: {e}")
                    continue

        return images, labels

    images, labels = load_dataset(path, classes)
    print(f"Total images loaded: {len(images)}")

    # Convert to numpy arrays
    # Check if images list is empty before converting to numpy array
    if not images:
        print("No images were loaded. Please check the dataset path and file patterns.")
        return np.array([]), np.array([]), np.array([]), np.array([]), classes

    images = np.array(images)
    labels = np.array(labels)

    # Split dataset (80-20)
    dataset_size = len(images)
    indices = np.random.permutation(dataset_size)
    train_size = int(0.8 * dataset_size)

    train_images = images[indices[:train_size]]
    train_labels = labels[indices[:train_size]]
    test_images = images[train_size:]
    test_labels = labels[train_size:]

    return train_images, train_labels, test_images, test_labels, classes

# Load the dataset
train_images, train_labels, test_images, test_labels, classes = load_and_preprocess_dataset()

# Step 4: Create the MLTSDC model according to paper specifications
def create_mltsdc_model(img_size=256, patch_size=16, num_classes=5, num_heads=2, depth=6):
    """
    Create Multi Level Transformer Based Sugarcane Disease Classifier
    Based on the paper's architecture with Patch Encoder and AI Blocks
    """
    inputs = keras.Input(shape=(img_size, img_size, 3))

    # Patch Encoder (Eq. 1-3)
    num_patches = (img_size // patch_size) ** 2
    embed_dim = 3 * patch_size * patch_size  # 768 as per paper

    # Create patches using Conv2D
    patches = layers.Conv2D(
        filters=embed_dim,
        kernel_size=patch_size,
        strides=patch_size,
        padding='valid',
        name='patch_encoder'
    )(inputs)
    patches = layers.Reshape((num_patches, embed_dim))(patches)

    # Positional Encoding (Eq. 2 - incremental embedding)
    positions = tf.range(start=0, limit=num_patches, delta=1)
    position_embedding = layers.Embedding(
        input_dim=num_patches,
        output_dim=embed_dim,
        name='position_embedding'
    )(positions)

    # Add positional encoding to patches
    x = patches + position_embedding

    # Transformer Blocks (AI Blocks)
    for i in range(depth):
        # Layer Normalization
        x1 = layers.LayerNormalization(epsilon=1e-6, name=f'layer_norm1_{i}')(x)

        # Multi-Head Attention (Eq. 4-7)
        attention_output = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim // num_heads,
            name=f'multi_head_attention_{i}'
        )(x1, x1)

        # First residual connection
        x2 = layers.Add(name=f'add1_{i}')([attention_output, x])

        # Layer Normalization
        x3 = layers.LayerNormalization(epsilon=1e-6, name=f'layer_norm2_{i}')(x2)

        # Feed Forward Network (Eq. 10)
        ff_dense1 = layers.Dense(2048, activation='relu', name=f'ff_dense1_{i}')(x3)
        ff_dense2 = layers.Dense(embed_dim, name=f'ff_dense2_{i}')(ff_dense1)


        # Second residual connection
        x = layers.Add(name=f'add2_{i}')([ff_dense2, x2])

    # Final Layer Normalization
    x = layers.LayerNormalization(epsilon=1e-6, name='final_layer_norm')(x)

    # Global Average Pooling
    x = layers.GlobalAveragePooling1D(name='global_avg_pooling')(x)

    # Dense layers for classification (Eq. 12)
    x = layers.Dense(512, activation='gelu', name='dense_1')(x)
    x = layers.Dropout(0.2, name='dropout_1')(x)
    x = layers.Dense(256, activation='gelu', name='dense_2')(x)
    x = layers.Dropout(0.2, name='dropout_2')(x)

    # Output layer
    outputs = layers.Dense(num_classes, activation='softmax', name='output')(x)

    model = keras.Model(inputs=inputs, outputs=outputs, name='MLTSDC')
    return model

# Step 5: Create datasets with proper preprocessing
def create_datasets(train_images, train_labels, test_images, test_labels, batch_size=32):
    def preprocess_fn(image, label):
        # Resize to 256x256 as per paper
        image = tf.image.resize(image, [256, 256])
        # Normalize to [0,1] as per paper
        image = image / 255.0
        return image, label

    # Create TensorFlow datasets
    train_dataset = tf.data.Dataset.from_tensor_slices((train_images, train_labels))
    test_dataset = tf.data.Dataset.from_tensor_slices((test_images, test_labels))

    # Apply preprocessing
    train_dataset = train_dataset.map(preprocess_fn, num_parallel_calls=tf.data.AUTOTUNE)
    test_dataset = test_dataset.map(preprocess_fn, num_parallel_calls=tf.data.AUTOTUNE)

    # Batch and optimize
    train_dataset = train_dataset.shuffle(1000).batch(batch_size).prefetch(tf.data.AUTOTUNE)
    test_dataset = test_dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)

    return train_dataset, test_dataset

# Create datasets
train_dataset, test_dataset = create_datasets(train_images, train_labels, test_images, test_labels)

# Step 6: Create and compile the main model
print("Creating MLTSDC model...")
mltsdc_model = create_mltsdc_model(num_classes=5)

# Compile with Adam optimizer as per paper Table 2
mltsdc_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Print model summary
mltsdc_model.summary()

# Step 7: Train the model
print("Training MLTSDC model...")
history = mltsdc_model.fit(
    train_dataset,
    epochs=5,
    validation_data=test_dataset,
    callbacks=[
        keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(patience=5, factor=0.5)
    ]
)

# Step 8: Evaluate the model
test_loss, test_accuracy = mltsdc_model.evaluate(test_dataset)
print(f"Test Accuracy: {test_accuracy*100:.2f}%")

# Step 9: Create the two-level system (Binary + Disease models)
def create_binary_model():
    """Level 1: Healthy vs Unhealthy"""
    base_model = create_mltsdc_model(num_classes=2)
    return base_model

def create_disease_model():
    """Level 2: Disease classification (4 classes)"""
    base_model = create_mltsdc_model(num_classes=4)
    return base_model

# Prepare binary labels (Level 1)
train_binary_labels = np.where(train_labels == 0, 0, 1)  # 0=Healthy, 1=Unhealthy
test_binary_labels = np.where(test_labels == 0, 0, 1)

# Prepare disease labels (Level 2) - only unhealthy samples
train_unhealthy_idx = np.where(train_labels > 0)[0]
train_disease_images = train_images[train_unhealthy_idx]
train_disease_labels = train_labels[train_unhealthy_idx] - 1  # Convert to 0-3

test_unhealthy_idx = np.where(test_labels > 0)[0]
test_disease_images = test_images[test_unhealthy_idx]
test_disease_labels = test_labels[test_unhealthy_idx] - 1

# Create datasets for both levels
train_binary_dataset, test_binary_dataset = create_datasets(
    train_images, train_binary_labels, test_images, test_binary_labels
)

train_disease_dataset, test_disease_dataset = create_datasets(
    train_disease_images, train_disease_labels, test_disease_images, test_disease_labels
)

# Create and train binary model
print("Training Binary Model (Level 1)...")
binary_model = create_binary_model()
binary_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

binary_history = binary_model.fit(
    train_binary_dataset,
    epochs=2,
    validation_data=test_binary_dataset,
    callbacks=[keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)]
)

# Create and train disease model
print("Training Disease Model (Level 2)...")
disease_model = create_disease_model()
disease_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

disease_history = disease_model.fit(
    train_disease_dataset,
    epochs=2,
    validation_data=test_disease_dataset,
    callbacks=[keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)]
)

# Step 10: Model quantization for mobile deployment
# Removed quantization due to compatibility issues

# Step 11: Convert to TensorFlow Lite
# Removed TFLite conversion due to removed quantization

print("MLTSDC implementation completed successfully!")

In [ ]:
# Configuration 1: learning rate 0.001, Adagrad
mltsdc_model_config1 = create_mltsdc_model(num_classes=5)
mltsdc_model_config1.compile(
    optimizer=keras.optimizers.Adagrad(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
print("Model Config 1 (Adagrad, 0.001) Summary:")
mltsdc_model_config1.summary()

# Configuration 2: learning rate 0.0001, Adadelta
mltsdc_model_config2 = create_mltsdc_model(num_classes=5)
mltsdc_model_config2.compile(
    optimizer=keras.optimizers.Adadelta(learning_rate=0.0001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
print("Model Config 2 (Adadelta, 0.0001) Summary:")
mltsdc_model_config2.summary()

# Configuration 3: learning rate 0.01, Nadam
mltsdc_model_config3 = create_mltsdc_model(num_classes=5)
mltsdc_model_config3.compile(
    optimizer=keras.optimizers.Nadam(learning_rate=0.01),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
print("Model Config 3 (Nadam, 0.01) Summary:")
mltsdc_model_config3.summary()

In [ ]:
# Prepare binary and disease datasets (already defined in original code)

# Configuration 1: learning rate 0.001, Adagrad
print("Training Binary Model Config 1 (Adagrad, 0.001)...")
binary_model_config1 = create_binary_model()
binary_model_config1.compile(
    optimizer=keras.optimizers.Adagrad(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
binary_history_config1 = binary_model_config1.fit(
    train_binary_dataset,
    epochs=2,
    validation_data=test_binary_dataset,
    callbacks=[keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)]
)

print("Training Disease Model Config 1 (Adagrad, 0.001)...")
disease_model_config1 = create_disease_model()
disease_model_config1.compile(
    optimizer=keras.optimizers.Adagrad(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
disease_history_config1 = disease_model_config1.fit(
    train_disease_dataset,
    epochs=2,
    validation_data=test_disease_dataset,
    callbacks=[keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)]
)

# Configuration 2: learning rate 0.0001, Adadelta
print("Training Binary Model Config 2 (Adadelta, 0.0001)...")
binary_model_config2 = create_binary_model()
binary_model_config2.compile(
    optimizer=keras.optimizers.Adadelta(learning_rate=0.0001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
binary_history_config2 = binary_model_config2.fit(
    train_binary_dataset,
    epochs=2,
    validation_data=test_binary_dataset,
    callbacks=[keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)]
)

print("Training Disease Model Config 2 (Adadelta, 0.0001)...")
disease_model_config2 = create_disease_model()
disease_model_config2.compile(
    optimizer=keras.optimizers.Adadelta(learning_rate=0.0001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
disease_history_config2 = disease_model_config2.fit(
    train_disease_dataset,
    epochs=2,
    validation_data=test_disease_dataset,
    callbacks=[keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)]
)

# Configuration 3: learning rate 0.01, Nadam
print("Training Binary Model Config 3 (Nadam, 0.01)...")
binary_model_config3 = create_binary_model()
binary_model_config3.compile(
    optimizer=keras.optimizers.Nadam(learning_rate=0.01),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
binary_history_config3 = binary_model_config3.fit(
    train_binary_dataset,
    epochs=2,
    validation_data=test_binary_dataset,
    callbacks=[keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)]
)

print("Training Disease Model Config 3 (Nadam, 0.01)...")
disease_model_config3 = create_disease_model()
disease_model_config3.compile(
    optimizer=keras.optimizers.Nadam(learning_rate=0.01),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
disease_history_config3 = disease_model_config3.fit(
    train_disease_dataset,
    epochs=2,
    validation_data=test_disease_dataset,
    callbacks=[keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)]
)

In [ ]:
import matplotlib.pyplot as plt

# Assuming history objects are available from the model training
# Replace the manual loss values with history.history['loss'] and history.history['val_loss'] if available

# 1. Plot for Main Model (MLTSDC)
main_train_loss = [1.0539, 0.5533, 0.5226, 0.5350, 0.5176]  # Training loss from epochs 1-5
main_val_loss = [0.1958, 0.1846, 0.1720, 0.3526, 0.2443]    # Validation loss from epochs 1-5
epochs_main = range(1, len(main_train_loss) + 1)

plt.figure(figsize=(10, 6))
plt.plot(epochs_main, main_train_loss, 'b-', label='Training Loss')
plt.plot(epochs_main, main_val_loss, 'r-', label='Validation Loss')
plt.title('Training and Validation Loss - Main Model (MLTSDC)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

# 2. Plot for Binary Model (Level 1)
binary_train_loss = [0.9823, 0.5113]  # Training loss from epochs 1-2
binary_val_loss = [0.1701, 0.4708]    # Validation loss from epochs 1-2
epochs_binary = range(1, len(binary_train_loss) + 1)

plt.figure(figsize=(10, 6))
plt.plot(epochs_binary, binary_train_loss, 'b-', label='Training Loss')
plt.plot(epochs_binary, binary_val_loss, 'r-', label='Validation Loss')
plt.title('Training and Validation Loss - Binary Model (Level 1)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

# 3. Plot for Disease Model (Level 2)
disease_train_loss = [0.1359, 3.4294e-08]  # Training loss from epochs 1-2
disease_val_loss = [0.0000e+00, 0.0000e+00]  # Validation loss from epochs 1-2
epochs_disease = range(1, len(disease_train_loss) + 1)

plt.figure(figsize=(10, 6))
plt.plot(epochs_disease, disease_train_loss, 'b-', label='Training Loss')
plt.plot(epochs_disease, disease_val_loss, 'r-', label='Validation Loss')
plt.title('Training and Validation Loss - Disease Model (Level 2)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

## Synthetic Data (GenAI) + Post-Processing

This section connects your prompt bank (`prompt.csv`) with a simple generation loop and the repository script `post_process.py`.

Recommended convention while generating images:
- Save each raw generated image as **`<ID>.png`** or **`<ID>.jpg`** (where `ID` comes from `prompt.csv`).
- Put all raw images into a single folder, e.g. `raw_synthetic/`.

Then `post_process.py` will:
- convert to RGB
- center-crop to square (optional)
- resize to 256x256
- save as JPEG quality 90
- organize into folders by `Folder/Severity/Climate` from `prompt.csv`


In [ ]:
from pathlib import Path
import csv
import importlib.util
import subprocess
import sys

PROMPT_CSV = Path('prompt.csv')
RAW_DIR = Path('raw_synthetic')
PROCESSED_DIR = Path('data') / 'synthetic'

# Ensure common dependencies (helpful outside Kaggle).
if importlib.util.find_spec('PIL') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pillow'])

if not PROMPT_CSV.exists():
    raise FileNotFoundError(f'Missing {PROMPT_CSV}. Run from repo root or adjust path.')

with PROMPT_CSV.open('r', newline='', encoding='utf-8-sig') as f:
    rows = list(csv.DictReader(f))

print('prompts:', len(rows))
print('example row keys:', list(rows[0].keys()) if rows else [])
print('example prompt snippet:', (rows[0].get('Prompt','')[:120] + '...') if rows else '')

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
print('raw dir:', RAW_DIR.resolve())
print('processed dir:', PROCESSED_DIR.resolve())


In [ ]:
from PIL import Image

# NOTE: Plug in your generator here (Gemini / SD / FLUX / etc.).
# This stub is intentionally not calling any external API.

def generate_image(prompt: str) -> Image.Image:
    raise NotImplementedError('Connect your image generator here and return a PIL.Image.Image')

# Example loop (disabled by default):
# for row in rows:
#     img = generate_image(row['Prompt'])
#     out_path = RAW_DIR / f"{int(row['ID'])}.png"
#     img.save(out_path)
#     print('saved', out_path)


In [ ]:
import subprocess
import sys
from pathlib import Path

# Post-process raw images into the exact training format and folder layout.
# Files in RAW_DIR should be named like: 1.png, 2.jpg, ... so the ID maps to prompt.csv rows.

manifest = Path('data') / 'synthetic_manifest.csv'

cmd = [
    sys.executable, 'post_process.py',
    '--input', str(RAW_DIR),
    '--output', str(PROCESSED_DIR),
    '--prompts', str(PROMPT_CSV),
    '--manifest', str(manifest),
    '--strict-id-mapping',
]

print('Running:', ' '.join(cmd))
subprocess.check_call(cmd)
print('Manifest:', manifest)


In [ ]:
from pathlib import Path

# Quick check: count processed JPGs by folder
base = PROCESSED_DIR
counts = {}
for p in base.rglob('*.jpg'):
    rel = p.relative_to(base)
    key = str(rel.parts[0]) if rel.parts else 'unknown'
    counts[key] = counts.get(key, 0) + 1

print('Processed images by Pair_Code folder:')
for k in sorted(counts):
    print(k, counts[k])

# Expected: 4 folders (A/B/C/D) once you generate all 48 prompts.
